# WP4v2 — Notebook 2 : Entraînement de f_θ

f_θ apprend : `mean(z_mae)` ∈ ℝ^1024 → `cls_clip` ∈ ℝ^1024

**A l'inférence**, f_θ sera appliqué à chaque z_i individuellement
(pas seulement à la moyenne) — les 196 tokens projetés passeront
ensuite dans le MLP connector de LLaVA pour rejoindre l'espace LLM.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')


In [ ]:
data_train = torch.load('wp4v2_pairs_train.pt')
data_val   = torch.load('wp4v2_pairs_val.pt')

# Normalisation z_mae (stats train uniquement)
z_mae_mean = data_train['z_mae'].mean(dim=0)
z_mae_std  = data_train['z_mae'].std(dim=0).clamp(min=1e-6)
torch.save({'mean': z_mae_mean, 'std': z_mae_std}, 'wp4v2_norm_stats.pt')

z_train = (data_train['z_mae'] - z_mae_mean) / z_mae_std
z_val   = (data_val['z_mae']   - z_mae_mean) / z_mae_std

BATCH_SIZE   = 256
train_loader = DataLoader(TensorDataset(z_train, data_train['cls_clip']),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(TensorDataset(z_val,   data_val['cls_clip']),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Train : {len(z_train)} | Val : {len(z_val)}')


In [ ]:
class ProjectionMLP(nn.Module):
    """
    f_theta : MAE (1024) -> espace CLIP (1024)
    Meme dimension source et cible — changement de base dans ℝ^1024.
    BatchNorm -> Linear(1024->1024) -> GELU -> Dropout(0.3) -> LayerNorm -> Linear(1024->1024) -> L2 norm
    """
    def __init__(self, dim=1024, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(dim),
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(dim),
            nn.Linear(dim, dim),
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

model = ProjectionMLP().to(DEVICE)
print(model)
print(f'Parametres : {sum(p.numel() for p in model.parameters()):,}')


In [ ]:
N_EPOCHS, LR, WEIGHT_DECAY, PATIENCE = 200, 1e-3, 1e-3, 20
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
history   = {'train_loss': [], 'val_loss': [], 'val_cos': []}
best_cos, best_epoch, patience_counter = 0., 0, 0

def cosine_loss(a, b): return (1 - F.cosine_similarity(a, b)).mean()

def eval_epoch(loader):
    model.eval()
    tl, tc, n = 0., 0., 0
    with torch.no_grad():
        for z, c in loader:
            z, c = z.to(DEVICE), c.to(DEVICE)
            p = model(z)
            tl += cosine_loss(p, c).item() * z.shape[0]
            tc += F.cosine_similarity(p, c).mean().item() * z.shape[0]
            n  += z.shape[0]
    return tl/n, tc/n

for epoch in range(N_EPOCHS):
    model.train()
    tl, n = 0., 0
    for z, c in train_loader:
        z, c = z.to(DEVICE), c.to(DEVICE)
        optimizer.zero_grad()
        loss = cosine_loss(model(z), c)
        loss.backward()
        optimizer.step()
        tl += loss.item() * z.shape[0]; n += z.shape[0]
    scheduler.step()

    train_loss = tl/n
    val_loss, val_cos = eval_epoch(val_loader)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_cos'].append(val_cos)

    if val_cos > best_cos:
        best_cos, best_epoch, patience_counter = val_cos, epoch+1, 0
        torch.save(model.state_dict(), 'wp4v2_projection_best.pt')
    else:
        patience_counter += 1

    if (epoch+1) % 20 == 0:
        print(f'Epoch {epoch+1:3d} | train={train_loss:.4f} | val={val_loss:.4f} | '
              f'cos={val_cos:.4f} | patience={patience_counter}/{PATIENCE}')
    if patience_counter >= PATIENCE:
        print(f'Early stopping a l epoch {epoch+1}'); break

print(f'Meilleur modele : epoch {best_epoch}, val_cos={best_cos:.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].axvline(best_epoch-1, color='red', linestyle='--', alpha=0.5, label=f'best={best_epoch}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss cosinus')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['val_cos'], color='darkorange')
axes[1].axvline(best_epoch-1, color='red', linestyle='--', alpha=0.5, label=f'best={best_cos:.4f}')
axes[1].axhline(1.0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Cos. sim.')
axes[1].set_title('Alignement MAE -> CLS CLIP'); axes[1].legend()
plt.tight_layout()
plt.savefig('wp4v2_training_curves.png', dpi=150)
plt.show()
